In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Assumption: notebook is run from scrapers/game_stat_scraper/notebooks/
VIS_DIR = Path.cwd()
GAME_SCRAPER_ROOT = VIS_DIR.parent                  # .../scrapers/game_stat_scraper
PROJECT_ROOT = GAME_SCRAPER_ROOT.parent.parent     # project repo root


OUT_DIR = PROJECT_ROOT / "README_ASSETS"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# Inputs
DATA_ROOT = GAME_SCRAPER_ROOT / "data"
FINAL_CSV = DATA_ROOT / "final_goheels_with_features.csv"

In [ ]:
def save_png(name: str, fig=None, dpi=300):
    if fig is None:
        fig = plt.gcf()
    path = OUT_DIR / f"{name}.png"
    fig.savefig(path, format="png", dpi=dpi, bbox_inches="tight", facecolor="white")
    print("Saved:", path)

In [ ]:

# Load data

df = pd.read_csv(FINAL_CSV)

# Basic cleaning
df["attendance"] = pd.to_numeric(df["attendance"], errors="coerce")
df = df.loc[df["attendance"].notna()].copy()

# Promotion cleanup (metadata)
if "promotion" in df.columns:
    df["promotion"] = (
        df["promotion"]
        .fillna("None")
        .astype(str)
        .str.strip()
    )
else:
    df["promotion"] = "None"

# Convenience: list of engineered NCAA features
feat_cols = [c for c in df.columns if c.startswith("unc_") or c.startswith("opp_")]
numeric_feat_cols = df[feat_cols].select_dtypes(include=[np.number]).columns.tolist()


# Plot 1 — Correlation of engineered stats with attendance
corr = df[numeric_feat_cols].corrwith(df["attendance"]).dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(20)

fig = plt.figure(figsize=(10, 7))
plt.barh(top.index[::-1], top.values[::-1])
plt.axvline(0)
plt.title("Top engineered feature correlations with attendance (Pearson)")
plt.xlabel("Correlation with attendance")
plt.tight_layout()

save_png("plot_01_top_feature_corr", fig)  # <-- SAVE BEFORE SHOW
plt.show()
plt.close(fig)  # keeps notebook clean


# Plot 2 — Attendance vs UNC recent scoring (binned L10 feature)
x = "unc_bat_r_pg_l10"
tmp = df.loc[df[x].notna(), ["attendance", x]].copy()

# Quantile bins for interpretability
tmp["bin"] = pd.qcut(tmp[x], q=6, duplicates="drop")

bin_stats = (
    tmp.groupby("bin")
    .agg(
        mean_att=("attendance", "mean"),
        n=("attendance", "size"),
        x_min=(x, "min"),
        x_max=(x, "max"),
    )
    .reset_index(drop=True)
)

labels = [
    f"{a:.2f}–{b:.2f}\n(n={n})"
    for a, b, n in zip(bin_stats["x_min"], bin_stats["x_max"], bin_stats["n"])
]

fig = plt.figure(figsize=(10, 5))
plt.bar(range(len(bin_stats)), bin_stats["mean_att"])
plt.xticks(range(len(bin_stats)), labels, rotation=0)
plt.title("Mean attendance by UNC runs per game (L10, pre-game)")
plt.ylabel("Mean attendance")
plt.xlabel("UNC recent scoring (quantile bins)")
plt.tight_layout()
save_png("plot_02_attendance_by_unc_runs_l10", fig)
plt.show()
plt.close(fig)


# Plot 3 — “Good matchup” proxy using UNC + opponent features
# (excitement index from combined offense/HR + competitiveness)
needed = [
    "unc_bat_r_pg_l10",
    "opp_bat_r_pg_l10",
    "unc_bat_hr_pg_l10",
    "opp_bat_hr_pg_l10",
]

tmp = df.loc[df[needed].notna().all(axis=1), ["attendance"] + needed].copy()

tmp["combined_runs"] = tmp["unc_bat_r_pg_l10"] + tmp["opp_bat_r_pg_l10"]
tmp["combined_hr"] = tmp["unc_bat_hr_pg_l10"] + tmp["opp_bat_hr_pg_l10"]
tmp["competitiveness"] = -np.abs(
    tmp["unc_bat_r_pg_l10"] - tmp["opp_bat_r_pg_l10"]
)  # closer teams => higher value

# z-score each component so we aren't arbitrarily weighting raw scales
def z(s):
    std = s.std(ddof=0)
    return (s - s.mean()) / (std if std != 0 else 1.0)

tmp["matchup_excitement_index"] = (
    z(tmp["combined_runs"])
    + z(tmp["combined_hr"])
    + z(tmp["competitiveness"])
)

# Bin index into quintiles and plot mean attendance
tmp["exc_bin"] = pd.qcut(tmp["matchup_excitement_index"], q=5, duplicates="drop")

exc_stats = (
    tmp.groupby("exc_bin")
    .agg(
        mean_att=("attendance", "mean"),
        n=("attendance", "size"),
        idx_min=("matchup_excitement_index", "min"),
        idx_max=("matchup_excitement_index", "max"),
    )
    .reset_index(drop=True)
)

labels = [
    f"{a:.2f}–{b:.2f}\n(n={n})"
    for a, b, n in zip(exc_stats["idx_min"], exc_stats["idx_max"], exc_stats["n"])
]

fig = plt.figure(figsize=(10, 5))
plt.bar(range(len(exc_stats)), exc_stats["mean_att"])
plt.xticks(range(len(exc_stats)), labels)
plt.title("Mean attendance by 'matchup excitement' (UNC + opponent, pre-game)")
plt.ylabel("Mean attendance")
plt.xlabel("Excitement index quintiles (higher = more offense/HR + more competitive)")
plt.tight_layout()
save_png("plot_03_matchup_excitement", fig)
plt.show()
plt.close(fig)


# Plot 4 — Interaction with existing metadata: Promotion × performance tier
# (Does promotion matter more when UNC is playing well?)
x = "unc_bat_r_pg_l10"
tmp = df.loc[df[x].notna(), ["attendance", x, "promotion"]].copy()

# Use top promotions by frequency so plot stays readable
top_promos = tmp["promotion"].value_counts().head(8).index.tolist()
tmp = tmp.loc[tmp["promotion"].isin(top_promos)].copy()

# Define "low / mid / high" performance buckets using terciles
tmp["perf_tier"] = pd.qcut(
    tmp[x],
    q=3,
    labels=["Low recent UNC offense", "Mid", "High recent UNC offense"]
)

g = (
    tmp.groupby(["promotion", "perf_tier"])
    .agg(
        mean_att=("attendance", "mean"),
        n=("attendance", "size")
    )
    .reset_index()
)

# Pivot to wide for side-by-side bars
pivot = (
    g.pivot(index="promotion", columns="perf_tier", values="mean_att")
    .reindex(top_promos)
)

xpos = np.arange(len(pivot.index))
width = 0.25

fig = plt.figure(figsize=(12, 5))
plt.bar(xpos - width, pivot.iloc[:, 0], width, label=pivot.columns[0])
plt.bar(xpos, pivot.iloc[:, 1], width, label=pivot.columns[1])
plt.bar(xpos + width, pivot.iloc[:, 2], width, label=pivot.columns[2])

plt.xticks(xpos, pivot.index, rotation=30, ha="right")
plt.ylabel("Mean attendance")
plt.title("Mean attendance by promotion type, split by UNC recent offense (L10, pre-game)")
plt.legend()
plt.tight_layout()
save_png("plot_04_promo_x_perf_tier", fig)
plt.show()
plt.close(fig)

The NCAA-derived performance variables were engineered as **pre-game rolling features** (implemented with `shift(1)` + rolling windows) to avoid label leakage. In this initial EDA, several engineered “form” signals show meaningful associations with attendance: higher recent **home-run rates** and **run scoring** (for both UNC and the opponent) tend to align with higher attendance, and the **matchup excitement index** (combined offense/HR + competitiveness) increases monotonically across quintiles. This suggests fans respond not just to UNC quality, but also to the perceived attractiveness of the opponent and the likelihood of a compelling game. The binned UNC runs-per-game plot shows a clear upward pattern with recent scoring, with some indication of **diminishing returns** at the very top end (a potential nonlinearity/plateau worth modeling). Promotion effects remain important, but the promotion-by-performance split suggests promotions may **amplify** already strong demand (mid/high recent offense tiers) rather than fully substituting for weak on-field form. Some pitching/discipline style features appear negatively correlated with attendance in the simple correlation screen, which likely reflects confounding with season timing, opponent mix, and other metadata. So these correlations should be treated as exploratory rather than causal.

The current **L10** window size (10-game rolling) was selected **arbitrarily** as a first pass to validate the end-to-end pipeline (scrape → cache → standardize → crosswalk → feature build). Next, we will systematically tune window sizes (e.g., **L3/L5/L10/L15**), evaluate alternative lag structures, and test matchup formulations in interpretable classical models with controls for seasonality, weather, weekend, and promotions to identify the most predictive and stable attendance drivers.